# 01 - Dataset Exploration & Profiling
## Customer Support on Twitter (`thoughtvector/customer-support-on-twitter`)

This notebook provides reproducible data inspection, profiling, and candidate brand analysis for the Hiver SDE Intern AI Customer Support Agent assignment.

### Objectives
1. Inspect raw dataset files, formats, sizes, and row counts.
2. Profile column schema, data types, missing values, and ID uniqueness.
3. Understand inbound vs. outbound tweet representations and author taxonomy.
4. Unpack conversation reconstruction logic via graph linkages (`tweet_id`, `in_response_to_tweet_id`, `response_tweet_id`).
5. Extract and aggregate detailed statistics for all 108 customer support brands.
6. Objectively evaluate and rank candidate brands for the customer support agent task.



## 1. Environment Setup & File Inspection


In [ ]:
import os
import re
import json
import time
from collections import Counter, defaultdict
import pandas as pd
import numpy as np

RAW_DIR = os.path.join("..", "data", "raw")
TWCS_PATH = os.path.join(RAW_DIR, "twcs", "twcs.csv")
SAMPLE_PATH = os.path.join(RAW_DIR, "sample.csv")

print("Files in data/raw:")
for root, dirs, files in os.walk(RAW_DIR):
    for f in files:
        fpath = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  - {fpath} ({size_mb:.2f} MB)")



## 2. Sample Inspection & Schema Verification


In [ ]:
df_sample = pd.read_csv(SAMPLE_PATH)
print("Sample Shape:", df_sample.shape)
print("\nColumn Data Types:")
print(df_sample.dtypes)
print("\nFirst 5 Rows:")
df_sample.head()



## 3. Large-Scale Chunked Profiling
Because `twcs.csv` is ~516 MB with 2.8+ million rows, we profile the dataset using chunked streaming to prevent memory exhaustion.



In [ ]:
chunk_size = 350000
total_rows = 0
null_counts = defaultdict(int)
seen_tweet_ids = set()
duplicate_ids = 0
customer_authors = set()
brand_authors = set()
min_date = None
max_date = None

t0 = time.time()
print("Streaming twcs.csv in chunks...")

for i, chunk in enumerate(pd.read_csv(TWCS_PATH, chunksize=chunk_size, low_memory=False)):
    total_rows += len(chunk)
    
    # Track missing values
    for col in chunk.columns:
        null_counts[col] += int(chunk[col].isnull().sum())
        
    # Check duplicate tweet IDs
    for tid in chunk['tweet_id'].values:
        if tid in seen_tweet_ids:
            duplicate_ids += 1
        else:
            seen_tweet_ids.add(tid)
            
    # Track author taxonomy
    cust_mask = chunk['inbound'] == True
    customer_authors.update(chunk.loc[cust_mask, 'author_id'].astype(str).unique())
    brand_authors.update(chunk.loc[~cust_mask, 'author_id'].astype(str).unique())
    
    # Track date ranges
    chunk_dates = pd.to_datetime(chunk['created_at'], errors='coerce', format='%a %b %d %H:%M:%S +0000 %Y')
    c_min, c_max = chunk_dates.min(), chunk_dates.max()
    if min_date is None or (pd.notnull(c_min) and c_min < min_date):
        min_date = c_min
    if max_date is None or (pd.notnull(c_max) and c_max > max_date):
        max_date = c_max
        
    print(f"  Chunk {i+1}: {total_rows:,} rows processed...")

print(f"\nProfiling complete in {time.time() - t0:.1f}s")
print(f"Total Rows: {total_rows:,}")
print(f"Unique Tweet IDs: {len(seen_tweet_ids):,} (Duplicates: {duplicate_ids})")
print(f"Date Range: {min_date} to {max_date}")
print(f"Unique Customer Authors: {len(customer_authors):,}")
print(f"Unique Brand Accounts: {len(brand_authors):,}")



## 4. Column Summary & Missing Value Analysis


In [ ]:
summary_data = []
for col in df_sample.columns:
    missing = null_counts[col]
    summary_data.append({
        "Column": col,
        "Data Type": str(df_sample[col].dtype),
        "Missing Values": f"{missing:,}",
        "Missing (%)": f"{missing / total_rows * 100:.2f}%"
    })

pd.DataFrame(summary_data)



## 5. Conversation Graph Reconstruction Logic
In TWCS, conversations are represented as directed trees or chains using two pointer columns:
- `in_response_to_tweet_id`: Points to the parent tweet. Missing (`NaN`) for conversation starter tweets.
- `response_tweet_id`: Points to child reply tweet(s). Can be comma-separated if multiple replies branch out.



In [ ]:
# Trace an example multi-turn conversation thread from the sample
sample_inbound_start = df_sample[df_sample['inbound'] & df_sample['in_response_to_tweet_id'].isnull()].iloc[0]
print("Thread Starter Customer Tweet:")
print(f"  ID: {sample_inbound_start['tweet_id']} | Text: {sample_inbound_start['text']}")
print(f"  Direct Response ID: {sample_inbound_start['response_tweet_id']}")

# Follow child
child_id = int(str(sample_inbound_start['response_tweet_id']).split(',')[0])
child_tweet = df_sample[df_sample['tweet_id'] == child_id]
if not child_tweet.empty:
    child_row = child_tweet.iloc[0]
    print(f"\nBrand Reply Tweet:")
    print(f"  ID: {child_row['tweet_id']} | Author: {child_row['author_id']} | Text: {child_row['text']}")



## 6. Per-Brand Aggregations & Detailed Statistics
We aggregate volume, initiating threads, DM deflection rates, link rates, and text lengths for each brand across the dataset.



In [ ]:
# Load compiled brand statistics generated from the chunked dataset scan
STATS_JSON = os.path.join("..", "reports", "brand_stats.json")

# If json not in reports, fallback to scratch or load
if not os.path.exists(STATS_JSON):
    # Alternative path if generated in scratch
    alt_path = os.path.join("..", "reports", "brand_stats.json")

# Display top 25 brands sorted by total tweet volume
with open(os.path.join("..", "reports", "brand_stats.json"), "r") as f:
    stats_data = json.load(f)

brands_df = pd.DataFrame(stats_data["brands"].values())
brands_df = brands_df.sort_values(by="total_tweets", ascending=False)
brands_df.head(25)



## 7. Multi-Criteria Candidate Brand Ranking
To select the best brand for the AI Support Agent assignment, we evaluate candidate brands across five objective operational criteria:
1. **Total Volume & Initiating Threads**: Sufficient sample size for retrieval indexing and evaluation splits.
2. **Actionable Resolution (Low DM Deflection)**: Brands that provide substantive in-tweet troubleshooting instead of immediate DM requests.
3. **Escalation Boundary**: A balanced ratio of self-serve issues vs. issues requiring human escalation.
4. **Text Richness**: Substantive query and response lengths.
5. **Language Consistency**: >99.5% English/ASCII content to avoid multilingual complexity.



In [ ]:
# Filter candidate brands with substantial volume (>25,000 tweets)
candidates = brands_df[brands_df['total_tweets'] >= 25000].copy()

# Compute an Actionability Score: penalizes extreme DM deflection (>70%) and awards balanced escalation (~20-50%)
# Ideal candidates have high volume, actionable troubleshooting, and clean English text
candidates['escalation_balance_score'] = 100 - abs(candidates['pct_outbound_dm'] - 35)

cols_display = [
    'brand', 'total_tweets', 'outbound_tweets', 'inbound_tweets', 
    'inbound_first_turn', 'unique_customers', 'pct_outbound_dm', 
    'pct_outbound_url', 'avg_outbound_char_len', 'avg_inbound_char_len'
]

candidates[cols_display].head(15)



## 8. Summary of Findings & Next Steps
- Dataset profiling has been completed objectively without model training or synthetic data.
- The raw dataset is strictly preserved without modification.
- A comprehensive profiling report is maintained in `reports/dataset_profile.md`.
- Final brand selection will be made collaboratively based on these objective metrics.

